# Step 1: Create Inverted Index

### 1.1 Parse each document in the collection to extract terms from the text fields.

I load the data from unzipped file A2 from folder documents_cs and documents_en to two dicts, indexed as {'Article_name': Article_txt}. If Article_name is not provided I name the article as " f'unnamed_article_no_{i}' ", where index i runs from 0. Then in the folowing analysis I work with these two dicts.

In [6]:
# Imports
import os
from lxml import etree
import re
#!pip install simplemma
import simplemma
from simplemma import lemmatize
from collections import defaultdict

In [2]:
def load_the_xml_data_from_file_to_dict(lang: str):
    if lang not in ['CZE', 'EN']:
        raise ValueError("Invalid language. Use 'CZE' or 'EN'.")
    
    Dict_of_ARTICLES = {}
    working_dir = os.getcwd()
    parser = etree.XMLParser(recover=True)

    i = 0  # index for unnamed articles

    if lang == 'CZE':
        folder = 'documents_cs'
        file_list = os.listdir(f'{working_dir}/A2/{folder}')
        file_list.remove('czech.dtd')

        title_tag = 'TITLE'
        text_tag = 'TEXT'

        for filename in file_list:
            file_path = f'{working_dir}/A2/{folder}/{filename}'
            tree = etree.parse(file_path, parser=parser)
            root = tree.getroot()

            for doc in root.iter('DOC'):
                title_elem = doc.find(title_tag)
                text_elem = doc.find(text_tag)

                title = title_elem.text.strip() if title_elem is not None and title_elem.text else f'unnamed_article_no_{i}'
                text = text_elem.text.strip() if text_elem is not None and text_elem.text else 'no_text_here'

                Dict_of_ARTICLES[title] = str(text)
                if title.startswith('unnamed_article_no_'):
                    i += 1

    elif lang == 'EN':
        folder = 'documents_en'
        file_list = os.listdir(f'{working_dir}/A2/{folder}')
        for f in ['latimes2002.dtd', 'README']:
            if f in file_list:
                file_list.remove(f)

        for filename in file_list:
            file_path = f'{working_dir}/A2/{folder}/{filename}'
            tree = etree.parse(file_path, parser=parser)
            root = tree.getroot()

            for doc in root.iter('DOC'):
                title_elem = doc.find('HD')
                # Collect <LD> and <TE> elements
                lead_texts = [ld.text.strip() for ld in doc.findall('LD') if ld.text]
                main_texts = [te.text.strip() for te in doc.findall('TE') if te.text]
                full_text = ' '.join(lead_texts + main_texts).strip() or 'no_text_here'

                if title_elem is not None and title_elem.text:
                    title = title_elem.text.strip()
                else:
                    title = f'unnamed_article_no_{i}'
                    i += 1

                Dict_of_ARTICLES[title] = str(full_text)

    return Dict_of_ARTICLES


In [3]:
CZE_Dict_of_ARTICLES = load_the_xml_data_from_file_to_dict(lang='CZE')
EN_Dict_of_ARTICLES = load_the_xml_data_from_file_to_dict(lang='EN')

### 1.2 Normalize the terms by converting them to lowercase, removing punctuation, and applying stemming or lemmatization (!).

I have decided to apply lemmatization instead of stemming, using library simplemma 1.1.2, documentation is available at: https://pypi.org/project/simplemma/

In [4]:
def normalize(lang:str):
    if lang not in ['CZE', 'EN']:
        raise ValueError("Invalid language. Use 'CZE' or 'EN'.")
    
    CZE_Dict_of_ARTICLES = load_the_xml_data_from_file_to_dict(lang=lang)
    EN_Dict_of_ARTICLES = load_the_xml_data_from_file_to_dict(lang=lang)
    
    # Set language and corresponding dictionary
    Dict_of_ARTICLES = EN_Dict_of_ARTICLES if lang == 'EN' else CZE_Dict_of_ARTICLES
    Dict_of_ARTICLES_processed = {}

    # Process articles
    for name, value in Dict_of_ARTICLES.items():
        lowercased_txt = value.lower()
        clean_text = re.sub(r'[\n\\]', ' ', lowercased_txt) # I also need to clean some symbols that I included in the text while loading the data.
        clean_text = re.sub(r'[^\w\s]', '', clean_text)

        tokenized_txt = clean_text.split()
        if lang == 'EN':
            tokens = [simplemma.lemmatize(t, lang='en') for t in tokenized_txt]
        elif lang == 'CZE':
            tokens = [simplemma.lemmatize(t, lang='cs') for t in tokenized_txt]

        Dict_of_ARTICLES_processed[name] = tokens

    return Dict_of_ARTICLES_processed
    


In [5]:
EN_Dict_of_ARTICLES_processed = normalize(lang='EN')
CZE_Dict_of_ARTICLES_processed = normalize(lang='CZE')

### 1.3 For each term, add the document IDs to the postings lists in lexicographical order.

In [7]:
def create_inverted_index_dict(lang: str):
    if lang not in ['CZE', 'EN']:
        raise ValueError("Invalid language. Use 'CZE' or 'EN'.")
    
    EN_Dict_of_ARTICLES_processed = normalize(lang='EN')
    CZE_Dict_of_ARTICLES_processed = normalize(lang='CZE')
    
    Inverted_index = defaultdict(list)
    Dict_of_ARTICLES = EN_Dict_of_ARTICLES_processed if lang == 'EN' else CZE_Dict_of_ARTICLES_processed

    for name, value in Dict_of_ARTICLES.items():
        # Handle invalid inputs (e.g., None or non-list)
        if not isinstance(value, list) or value is None:
            value = []
        
        # Create set of unique tokens to avoid duplicates
        unique_tokens = set(value)

        # Add document name to postings list for each term
        for term in unique_tokens:
            if term:  # Skip empty terms
                Inverted_index[term].append(name)

    # Sort postings lists lexicographically once at the end
    for term in Inverted_index:
        Inverted_index[term].sort()

    return dict(Inverted_index)

In [9]:
Inverted_index_CZE = create_inverted_index_dict(lang='CZE')
Inverted_index_EN = create_inverted_index_dict(lang='EN')

# Step 2: Implement Boolean Query Operators

Implement Boolean query operators for two terms: x AND y, x OR y, and x AND NOT y by iterating over the sorted postings lists.

In [ ]:
def boolean_and(postings_x: list, postings_y: list):
    """Returns the intersection of two sorted postings lists (x AND y)."""
    result = []
    i = 0  # Pointer for postings_x
    j = 0  # Pointer for postings_y
    
    while i < len(postings_x) and j < len(postings_y):
        if postings_x[i] == postings_y[j]:
            result.append(postings_x[i])
            i += 1
            j += 1
        elif postings_x[i] < postings_y[j]:
            i += 1
        else:
            j += 1
            
    return result

def boolean_or(postings_x: list, postings_y: list):
    """Returns the union of two sorted postings lists (x OR y)."""
    result = []
    i = 0  # Pointer for postings_x
    j = 0  # Pointer for postings_y
    
    while i < len(postings_x) and j < len(postings_y):
        if postings_x[i] == postings_y[j]:
            result.append(postings_x[i])
            i += 1
            j += 1
        elif postings_x[i] < postings_y[j]:
            result.append(postings_x[i])
            i += 1
        else:
            result.append(postings_y[j])
            j += 1
    
    # Append remaining elements
    result.extend(postings_x[i:])
    result.extend(postings_y[j:])
    
    return result

def boolean_and_not(postings_x: list, postings_y: list):
    """Returns documents in postings_x but not in postings_y (x AND NOT y)."""
    result = []
    i = 0  # Pointer for postings_x
    j = 0  # Pointer for postings_y
    
    while i < len(postings_x):
        if j >= len(postings_y):  # No more y documents to exclude
            result.append(postings_x[i])
            i += 1
        elif postings_x[i] == postings_y[j]:
            i += 1  # Skip documents in both lists
            j += 1
        elif postings_x[i] < postings_y[j]:
            result.append(postings_x[i])  # x document not in y
            i += 1
        else:
            j += 1  # Skip y document
            
    return result

def process_boolean_query(term_x: str, term_y: str, inverted_index: dict):
    """Processes a Boolean query for two terms and returns results for AND, OR, AND NOT."""
    # Get postings lists, default to empty list if term not found
    postings_x = inverted_index.get(term_x, [])
    postings_y = inverted_index.get(term_y, [])
    
    # Compute results for each operator
    return {
        'AND': boolean_and(postings_x, postings_y),
        'OR': boolean_or(postings_x, postings_y),
        'AND NOT': boolean_and_not(postings_x, postings_y)
    }

# Step 3: Process Boolean Queries

Process the Boolean queries for the provided topics separately for Czech and English and generate the results.

### 3.1 Parse the topic file to extract queries.

In [ ]:


inputs_CZE = 